# 07 — Consultas espaciales MT

Esta notebook se va a construir de a bloques para evitar respuestas largas y cuelgues durante la edición.

En este primer avance quedan cargadas las consultas 1 a 3: proximidad puntual y buffers simples.


## 1. Objetivo y prerequisitos

- Tener levantado PostGIS con `docker compose up -d postgis`.
- Haber ejecutado la notebook 06 para publicar `gis.mt_cables`, `gis.mt_postes` y `gis.mt_seccionadores`.
- Ejecutar las celdas en orden. Cada consulta deja el SQL visible antes del resultado.

## 2. Preparación de conexión

In [45]:
# pathlib permite ubicar la raíz del proyecto aunque Jupyter se abra desde otra carpeta.
from pathlib import Path

# os permite leer variables de entorno para parametrizar la conexión local a PostGIS.
import os

# json permite armar GeoJSON para visualizar resultados espaciales en el navegador.
import json

# escape permite insertar un documento HTML completo dentro de un iframe sin romper atributos.
from html import escape

# uuid evita repetir identificadores HTML cuando se muestran varios mapas en la misma notebook.
import uuid

# Decimal aparece cuando PostgreSQL devuelve columnas numeric; hay que convertirlo para GeoJSON/JavaScript.
from decimal import Decimal

# pandas se usa solo para mostrar resultados tabulares de forma cómoda en Jupyter.
import pandas as pd

# IPython.display permite insertar un mapa Leaflet como HTML dentro de la notebook.
from IPython.display import HTML, display

# psycopg es el driver moderno de PostgreSQL usado por el proyecto.
try:
    import psycopg
    PSYCOPG_DISPONIBLE = True
except ModuleNotFoundError:
    psycopg = None
    PSYCOPG_DISPONIBLE = False
    print('No está instalado el paquete psycopg en este kernel.')
    print('Ejecutar desde la raíz del proyecto: python3 -m pip install -r requirements.txt')
    print('Después de instalar, reiniciar el kernel de Jupyter y volver a ejecutar la notebook.')

# Buscamos la raíz del proyecto subiendo hasta encontrar archivos propios del repo.
RAIZ = Path.cwd()
while RAIZ.parent != RAIZ:
    if (RAIZ / 'docker-compose.yml').exists() and (RAIZ / 'scripts' / 'gis_sqlserver.sh').exists():
        break
    RAIZ = RAIZ.parent

# Cargamos .env local sin pisar variables ya definidas por el entorno.
def cargar_env_local() -> None:
    env_path = RAIZ / '.env'
    if not env_path.exists():
        return
    for line in env_path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip())

cargar_env_local()

DB_CONFIG = {
    'host': os.getenv('POSTGRES_HOST', 'localhost'),
    'port': int(os.getenv('POSTGRES_PORT', '5432')),
    'dbname': os.getenv('POSTGRES_DB', 'ceml_gis'),
    'user': os.getenv('POSTGRES_USER', 'ceml'),
    'password': os.getenv('POSTGRES_PASSWORD', 'ceml_admin_2026'),
}

SRID_PUBLICACION = 32721

# Esta función corta con una instrucción clara si falta el driver de PostgreSQL.
def exigir_psycopg() -> None:
    if not PSYCOPG_DISPONIBLE:
        raise RuntimeError(
            'Falta instalar psycopg en este kernel. '
            'Ejecutar: python -m pip install -r requirements.txt, '
            'reiniciar el kernel y volver a correr la notebook.'
        )

# Ejecuta una consulta SELECT y devuelve un DataFrame.
# El autocommit evita dejar transacciones abiertas durante ejercicios de solo lectura.
def ejecutar_sql(sql: str) -> pd.DataFrame:
    exigir_psycopg()
    with psycopg.connect(**DB_CONFIG, autocommit=True) as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
            columnas = [col.name for col in cur.description]
            filas = cur.fetchall()
    return pd.DataFrame(filas, columns=columnas)


# Convierte valores devueltos por PostgreSQL a tipos compatibles con JSON/JavaScript.
# Ejemplo: round(... )::numeric vuelve como Decimal en Python y json.dumps no lo puede serializar.
def valor_json(valor):
    if isinstance(valor, Decimal):
        return float(valor)
    return valor

# Ejecuta una consulta que debe traer una columna `geojson` en EPSG:4326.
# Devuelve una FeatureCollection para poder dibujarla en el mapa.
def ejecutar_geojson(sql: str) -> dict:
    exigir_psycopg()
    with psycopg.connect(**DB_CONFIG, autocommit=True) as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
            columnas = [col.name for col in cur.description]
            filas = cur.fetchall()

    features = []
    for fila in filas:
        registro = dict(zip(columnas, fila))
        geometria = registro.pop('geojson', None)
        if geometria is None:
            continue
        if isinstance(geometria, str):
            geometria = json.loads(geometria)
        properties = {clave: valor_json(valor) for clave, valor in registro.items()}
        features.append({
            'type': 'Feature',
            'geometry': geometria,
            'properties': properties,
        })
    return {'type': 'FeatureCollection', 'features': features}

# Muestra una FeatureCollection en un mapa Leaflet con OpenStreetMap.
# La geometría ya debe venir transformada a EPSG:4326 porque Leaflet trabaja con lon/lat.
# Usamos un iframe autónomo para que cada mapa inicialice Leaflet de forma aislada en JupyterLab.
def mostrar_mapa(feature_collection: dict, titulo: str = 'Mapa') -> None:
    features = feature_collection.get('features', [])
    if not features:
        display(HTML('<p><strong>Sin geometrías para mostrar en el mapa.</strong></p>'))
        return

    div_id = 'mapa_' + uuid.uuid4().hex
    geojson_js = json.dumps(feature_collection, ensure_ascii=False, default=str)
    mapa_html = f'''<!doctype html>
<html>
<head>
<meta charset="utf-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
<style>
    html, body {{ margin: 0; padding: 0; height: 100%; font-family: sans-serif; }}
    #titulo {{ font-weight: 600; padding: 8px 10px; }}
    #{div_id} {{ height: 520px; border-top: 1px solid #bbb; }}
    table {{ border-collapse: collapse; }}
    th, td {{ padding: 2px 6px; border-bottom: 1px solid #ddd; vertical-align: top; }}
</style>
</head>
<body>
<div id="titulo">{titulo}</div>
<div id="{div_id}"></div>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script>
(function() {{
    const data = {geojson_js};
    const map = L.map('{div_id}');
    L.tileLayer('https://tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png', {{
        maxZoom: 19,
        attribution: '&copy; OpenStreetMap contributors'
    }}).addTo(map);

    const layer = L.geoJSON(data, {{
        style: function(feature) {{
            const type = feature.geometry ? feature.geometry.type : '';
            if (type.includes('Polygon')) {{
                return {{color: '#d94801', weight: 2, fillOpacity: 0.20}};
            }}
            if (type.includes('LineString')) {{
                return {{color: '#08519c', weight: 4}};
            }}
            return {{color: '#08519c', weight: 2}};
        }},
        pointToLayer: function(feature, latlng) {{
            return L.circleMarker(latlng, {{radius: 6, color: '#08519c', fillColor: '#3182bd', fillOpacity: 0.85}});
        }},
        onEachFeature: function(feature, layer) {{
            const props = feature.properties || {{}};
            const rows = Object.entries(props)
                .map(([k, v]) => `<tr><th style="text-align:left;">${{k}}</th><td>${{v}}</td></tr>`)
                .join('');
            layer.bindPopup(`<table>${{rows}}</table>`);
        }}
    }}).addTo(map);

    try {{
        const bounds = layer.getBounds();
        if (bounds.isValid()) {{
            map.fitBounds(bounds, {{padding: [20, 20]}});
        }} else {{
            map.setView([-26.66128, -54.73465], 13);
        }}
    }} catch (error) {{
        map.setView([-26.66128, -54.73465], 13);
    }}
}})();
</script>
</body>
</html>'''

    iframe = f'''<iframe
        srcdoc="{escape(mapa_html, quote=True)}"
        width="100%"
        height="570"
        style="border: 1px solid #bbb; border-radius: 4px;"
        loading="lazy">
    </iframe>'''
    display(HTML(iframe))

print('Raíz del proyecto:', RAIZ)
print('Base local:', DB_CONFIG['dbname'])
print('Host local:', DB_CONFIG['host'])
print('Puerto local:', DB_CONFIG['port'])
print('Usuario local:', DB_CONFIG['user'])
print('SRID de publicación:', SRID_PUBLICACION)

Raíz del proyecto: /home/jovyan/work
Base local: ceml_gis
Host local: postgis
Puerto local: 5432
Usuario local: ceml
SRID de publicación: 32721


## 3. Verificación de capas GIS

In [46]:
# Verificamos que estén publicadas las capas finales que usan estas consultas.
SQL_VERIFICAR_CAPAS = """
SELECT
    table_schema,
    table_name
FROM information_schema.tables
WHERE table_schema = 'gis'
  AND table_name IN ('mt_cables', 'mt_postes', 'mt_seccionadores')
ORDER BY table_name;
"""

ejecutar_sql(SQL_VERIFICAR_CAPAS)

,table_schema,table_name
0,gis,mt_cables
1,gis,mt_postes
2,gis,mt_seccionadores


## 4. Consulta 1: Postes cercanos a un punto

**Consulta espacial:** Búsqueda por proximidad ST_DWithin  
**Función:** Buscar postes dentro de un radio determinado.  
**Consigna:** Obtener todos los postes ubicados hasta 500 metros del punto geográfico (-54.73465, -26.66128), mostrando su identificador, código CAD, cooperativa y distancia.

**Idea clave:** Busca postes cuya geometría esté a 500 metros o menos del punto indicado. Como las capas están en SRID 32721, el punto ingresado en lon/lat se transforma desde 4326 a 32721.

In [47]:
SQL_CONSULTA_01 = """
SELECT
    p.id,
    p.idcad,
    p.coop,
    p.capa_origen,
    round(
        ST_Distance(
            p.geom,
            ST_Transform(
                ST_SetSRID(ST_MakePoint(-54.73465, -26.66128), 4326),
                32721
            )
        )::numeric,
        2
    ) AS distancia_m
FROM gis.mt_postes p
WHERE ST_DWithin(
    p.geom,
    ST_Transform(
        ST_SetSRID(ST_MakePoint(-54.73465, -26.66128), 4326),
        32721
    ),
    500
)
ORDER BY distancia_m;
"""

print(SQL_CONSULTA_01)


SELECT
    p.id,
    p.idcad,
    p.coop,
    p.capa_origen,
    round(
        ST_Distance(
            p.geom,
            ST_Transform(
                ST_SetSRID(ST_MakePoint(-54.73465, -26.66128), 4326),
                32721
            )
        )::numeric,
        2
    ) AS distancia_m
FROM gis.mt_postes p
WHERE ST_DWithin(
    p.geom,
    ST_Transform(
        ST_SetSRID(ST_MakePoint(-54.73465, -26.66128), 4326),
        32721
    ),
    500
)
ORDER BY distancia_m;



In [48]:
ejecutar_sql(SQL_CONSULTA_01)

,id,idcad,coop,capa_origen,distancia_m
0,3,MTENS257953,MCARL,.00.MT_Postes,0.44
1,4,MTENS257954,MCARL,.00.MT_Postes,79.69
2,3021,MTENS257630,MCARL,.00.MT_Postes,79.83
3,3020,MTENS257631,MCARL,.00.MT_Postes,110.66
4,3022,MTENS257629,MCARL,.00.MT_Postes,154.00
5,5,MTENS257955,MCARL,.00.MT_Postes,159.37
6,3019,MTENS257632,MCARL,.00.MT_Postes,172.41
7,3023,MTENS257628,MCARL,.00.MT_Postes,233.06
8,6,MTENS257956,MCARL,.00.MT_Postes,239.15
9,3018,MTENS257633,MCARL,.00.MT_Postes,242.51


### Mapa consulta

Visualizamos los postes encontrados por la consulta de proximidad.

In [49]:
SQL_MAPA_01 = """
SELECT
    p.id,
    p.idcad,
    p.coop,
    round(
        ST_Distance(
            p.geom,
            ST_Transform(ST_SetSRID(ST_MakePoint(-54.73465, -26.66128), 4326), 32721)
        )::numeric,
        2
    ) AS distancia_m,
    ST_AsGeoJSON(ST_Transform(p.geom, 4326))::json AS geojson
FROM gis.mt_postes p
WHERE ST_DWithin(
    p.geom,
    ST_Transform(ST_SetSRID(ST_MakePoint(-54.73465, -26.66128), 4326), 32721),
    500
)
ORDER BY distancia_m;
"""

mostrar_mapa(ejecutar_geojson(SQL_MAPA_01), 'Consulta 1 — postes cercanos')

## 5. Consulta 2: Seccionadores cercanos a un punto

**Consulta espacial:** Proximidad ST_DWithin  
**Función:** Buscar seccionadores cercanos a una ubicación.  
**Consigna:** Listar los seccionadores ubicados hasta 1000 metros del punto (-54.73465, -26.66128).

**Idea clave:** Filtra los seccionadores que caen dentro de un radio de 1000 metros desde el punto dado y calcula la distancia exacta.

In [50]:
SQL_CONSULTA_02 = """
SELECT
    s.id,
    s.idcad,
    s.coop,
    round(
        ST_Distance(
            s.geom,
            ST_Transform(
                ST_SetSRID(ST_MakePoint(-54.73465, -26.66128), 4326),
                32721
            )
        )::numeric,
        2
    ) AS distancia_m
FROM gis.mt_seccionadores s
WHERE ST_DWithin(
    s.geom,
    ST_Transform(
        ST_SetSRID(ST_MakePoint(-54.73465, -26.66128), 4326),
        32721
    ),
    1000
)
ORDER BY distancia_m;
"""

print(SQL_CONSULTA_02)


SELECT
    s.id,
    s.idcad,
    s.coop,
    round(
        ST_Distance(
            s.geom,
            ST_Transform(
                ST_SetSRID(ST_MakePoint(-54.73465, -26.66128), 4326),
                32721
            )
        )::numeric,
        2
    ) AS distancia_m
FROM gis.mt_seccionadores s
WHERE ST_DWithin(
    s.geom,
    ST_Transform(
        ST_SetSRID(ST_MakePoint(-54.73465, -26.66128), 4326),
        32721
    ),
    1000
)
ORDER BY distancia_m;



In [51]:
ejecutar_sql(SQL_CONSULTA_02)

,id,idcad,coop,distancia_m
0,426,MTENS257702,MCARL,79.82
1,427,MTENS257697,MCARL,594.35
2,428,MTENS257614,MCARL,598.10
3,423,MTENS257713,MCARL,632.40
4,422,MTENS257714,MCARL,640.51
5,416,MTENS257720,MCARL,930.22


### Mapa consulta

Visualizamos los seccionadores cercanos al punto de referencia.

In [52]:
SQL_MAPA_02 = """
SELECT
    s.id,
    s.idcad,
    s.coop,
    round(
        ST_Distance(
            s.geom,
            ST_Transform(ST_SetSRID(ST_MakePoint(-54.73465, -26.66128), 4326), 32721)
        )::numeric,
        2
    ) AS distancia_m,
    ST_AsGeoJSON(ST_Transform(s.geom, 4326))::json AS geojson
FROM gis.mt_seccionadores s
WHERE ST_DWithin(
    s.geom,
    ST_Transform(ST_SetSRID(ST_MakePoint(-54.73465, -26.66128), 4326), 32721),
    1000
)
ORDER BY distancia_m;
"""

mostrar_mapa(ejecutar_geojson(SQL_MAPA_02), 'Consulta 2 — seccionadores cercanos')

## 6. Consulta 3: Cables que atraviesan un área de influencia

**Consulta espacial:** Intersección con buffer ST_Intersects + ST_Buffer  
**Función:** Buscar cables que pasan cerca de un punto.  
**Consigna:** Obtener los cables que intersectan un área circular de 300 metros alrededor del punto (-54.73465, -26.66128).

**Idea clave:** Primero genera un área circular de 300 metros alrededor del punto. Luego devuelve los cables cuya traza intersecta esa área.

In [53]:
SQL_CONSULTA_03 = """
SELECT
    c.id,
    c.idcad,
    c.coop,
    round(ST_Length(c.geom)::numeric, 2) AS largo_m
FROM gis.mt_cables c
WHERE ST_Intersects(
    c.geom,
    ST_Buffer(
        ST_Transform(
            ST_SetSRID(ST_MakePoint(-54.73465, -26.66128), 4326),
            32721
        ),
        300
    )
)
ORDER BY largo_m DESC;
"""

print(SQL_CONSULTA_03)


SELECT
    c.id,
    c.idcad,
    c.coop,
    round(ST_Length(c.geom)::numeric, 2) AS largo_m
FROM gis.mt_cables c
WHERE ST_Intersects(
    c.geom,
    ST_Buffer(
        ST_Transform(
            ST_SetSRID(ST_MakePoint(-54.73465, -26.66128), 4326),
            32721
        ),
        300
    )
)
ORDER BY largo_m DESC;



In [54]:
ejecutar_sql(SQL_CONSULTA_03)

,id,idcad,coop,largo_m
0,3014,MTENS257664,MCARL,87.80
1,3013,MTENS257665,MCARL,80.43
2,135,MTENS503043,MCARL,79.81
3,137,MTENS503045,MCARL,79.78
4,138,MTENS503046,MCARL,79.73
5,134,MTENS503042,MCARL,79.70
6,136,MTENS503044,MCARL,79.67
7,3009,MTENS257669,MCARL,76.18
8,3010,MTENS257668,MCARL,76.18
9,3011,MTENS257667,MCARL,76.18


### Mapa consulta

Visualizamos los cables que intersectan el buffer de 300 metros.

In [55]:
SQL_MAPA_03 = """
SELECT
    c.id,
    c.idcad,
    c.coop,
    round(ST_Length(c.geom)::numeric, 2) AS largo_m,
    ST_AsGeoJSON(ST_Transform(c.geom, 4326))::json AS geojson
FROM gis.mt_cables c
WHERE ST_Intersects(
    c.geom,
    ST_Buffer(
        ST_Transform(ST_SetSRID(ST_MakePoint(-54.73465, -26.66128), 4326), 32721),
        300
    )
)
ORDER BY largo_m DESC;
"""

mostrar_mapa(ejecutar_geojson(SQL_MAPA_03), 'Consulta 3 — cables en área de influencia')

# Bloque 2 — Relaciones entre capas

Este bloque cruza seccionadores, cables y postes para responder preguntas de relación espacial.

## Consulta 4: Cable más cercano a cada seccionador

**Consulta espacial:** Vecino más cercano  
**Función:** Asociar cada seccionador con el cable más próximo.  
**Consigna:** Para cada seccionador, obtener el cable más cercano y la distancia entre ambos.

**Idea clave:** Para cada seccionador busca el cable más cercano usando el operador espacial <->, útil para consultas de vecino más próximo.

In [56]:
SQL_CONSULTA_04 = """
SELECT
    s.id AS id_seccionador,
    s.idcad AS idcad_seccionador,
    c.id AS id_cable,
    c.idcad AS idcad_cable,
    round(ST_Distance(s.geom, c.geom)::numeric, 2) AS distancia_m
FROM gis.mt_seccionadores s
CROSS JOIN LATERAL (
    SELECT
        c.id,
        c.idcad,
        c.geom
    FROM gis.mt_cables c
    ORDER BY s.geom <-> c.geom
    LIMIT 1
) c
ORDER BY distancia_m;
"""

print(SQL_CONSULTA_04)


SELECT
    s.id AS id_seccionador,
    s.idcad AS idcad_seccionador,
    c.id AS id_cable,
    c.idcad AS idcad_cable,
    round(ST_Distance(s.geom, c.geom)::numeric, 2) AS distancia_m
FROM gis.mt_seccionadores s
CROSS JOIN LATERAL (
    SELECT
        c.id,
        c.idcad,
        c.geom
    FROM gis.mt_cables c
    ORDER BY s.geom <-> c.geom
    LIMIT 1
) c
ORDER BY distancia_m;



In [57]:
ejecutar_sql(SQL_CONSULTA_04)

,id_seccionador,idcad_seccionador,id_cable,idcad_cable,distancia_m
0,823,MTENS241130,2303,MTENS258927,0.00
1,822,MTENS241131,6479,MTENS239554,0.00
2,713,MTENS254564,4474,MTENS252745,0.00
3,715,MTENS254556,4492,MTENS252724,0.00
4,820,MTENS241134,245,MTENS267600,0.00
...,...,...,...,...,...
876,76,MTENS269368,6740,MTENS210712,15.91
877,854,MTENS219733,6540,MTENS219549,16.92
878,77,MTENS269366,6740,MTENS210712,29.86
879,78,MTENS269365,6740,MTENS210712,43.46


### Mapa consulta 4

Visualización espacial del resultado anterior.

In [58]:
SQL_MAPA_04 = """
SELECT
    s.id AS id_seccionador,
    s.idcad AS idcad_seccionador,
    c.id AS id_cable,
    c.idcad AS idcad_cable,
    round(ST_Distance(s.geom, c.geom)::numeric, 2) AS distancia_m,
    ST_AsGeoJSON(ST_Transform(ST_Collect(s.geom, c.geom), 4326))::json AS geojson
FROM gis.mt_seccionadores s
CROSS JOIN LATERAL (
    SELECT
        c.id,
        c.idcad,
        c.geom
    FROM gis.mt_cables c
    ORDER BY s.geom <-> c.geom
    LIMIT 1
) c
ORDER BY distancia_m;
"""

mostrar_mapa(ejecutar_geojson(SQL_MAPA_04), 'Consulta 4 — seccionador y cable más cercano')

## Consulta 5: Cantidad de postes cercanos a cada cable

**Consulta espacial:** Relación por cercanía ST_DWithin  
**Función:** Contar postes próximos a cada tramo de cable.  
**Consigna:** Listar los cables y contar cuántos postes se encuentran a menos de 10 metros de cada cable.

**Idea clave:** Relaciona cables y postes cuando la distancia entre ambos es menor o igual a 10 metros. Sirve para detectar qué tramos tienen más postes asociados espacialmente.

In [59]:
SQL_CONSULTA_05 = """
SELECT
    c.id,
    c.idcad,
    c.coop,
    count(p.id) AS postes_cercanos
FROM gis.mt_cables c
LEFT JOIN gis.mt_postes p
    ON ST_DWithin(p.geom, c.geom, 10)
GROUP BY c.id, c.idcad, c.coop
ORDER BY postes_cercanos DESC;
"""

print(SQL_CONSULTA_05)


SELECT
    c.id,
    c.idcad,
    c.coop,
    count(p.id) AS postes_cercanos
FROM gis.mt_cables c
LEFT JOIN gis.mt_postes p
    ON ST_DWithin(p.geom, c.geom, 10)
GROUP BY c.id, c.idcad, c.coop
ORDER BY postes_cercanos DESC;



In [60]:
ejecutar_sql(SQL_CONSULTA_05)

,id,idcad,coop,postes_cercanos
0,3538,MTENS256247,MCARL,7
1,179,MTENS273445,MCARL,5
2,204,MTENS273307,MCARL,5
3,202,MTENS273310,MCARL,5
4,205,MTENS273306,MCARL,5
...,...,...,...,...
7152,60,MTENS505644,MCARL,0
7153,178,MTENS459279,MCARL,0
7154,187,MTENS273361,MCARL,0
7155,223,MTENS272953,MCARL,0


### Mapa consulta 5

Visualización espacial del resultado anterior.

In [61]:
SQL_MAPA_05 = """
SELECT
    c.id,
    c.idcad,
    c.coop,
    count(p.id) AS postes_cercanos,
    ST_AsGeoJSON(ST_Transform(c.geom, 4326))::json AS geojson
FROM gis.mt_cables c
LEFT JOIN gis.mt_postes p
    ON ST_DWithin(p.geom, c.geom, 10)
GROUP BY c.id, c.idcad, c.coop, c.geom
ORDER BY postes_cercanos DESC
LIMIT 25;
"""

mostrar_mapa(ejecutar_geojson(SQL_MAPA_05), 'Consulta 5 — cables con más postes cercanos')

## Consulta 6: Postes dentro de un rectángulo de trabajo

**Consulta espacial:** Contención espacial ST_Within  
**Función:** Filtrar elementos dentro de un área rectangular.  
**Consigna:** Obtener los postes contenidos dentro de una ventana espacial definida en coordenadas UTM EPSG:32721.

**Idea clave:** ST_MakeEnvelope crea un rectángulo espacial. ST_Within verifica qué postes están completamente contenidos dentro de esa geometría.

In [62]:
SQL_CONSULTA_06 = """
SELECT
    p.id,
    p.idcad,
    p.coop,
    p.capa_origen
FROM gis.mt_postes p
WHERE ST_Within(
    p.geom,
    ST_MakeEnvelope(720000, 7040000, 725000, 7045000, 32721)
);
"""

print(SQL_CONSULTA_06)


SELECT
    p.id,
    p.idcad,
    p.coop,
    p.capa_origen
FROM gis.mt_postes p
WHERE ST_Within(
    p.geom,
    ST_MakeEnvelope(720000, 7040000, 725000, 7045000, 32721)
);



In [63]:
ejecutar_sql(SQL_CONSULTA_06)

,id,idcad,coop,capa_origen
0,62,MTENS269708,MCARL,.00.MT_Postes
1,1050,MTENS264350,MCARL,.00.MT_Postes
2,1051,MTENS264349,MCARL,.00.MT_Postes
3,1052,MTENS264348,MCARL,.00.MT_Postes
4,1053,MTENS264347,MCARL,.00.MT_Postes
...,...,...,...,...
122,2122,MTENS260376,MCARL,.00.MT_Postes
123,2123,MTENS260375,MCARL,.00.MT_Postes
124,2124,MTENS260374,MCARL,.00.MT_Postes
125,2125,MTENS260373,MCARL,.00.MT_Postes


### Mapa consulta 6

Visualización espacial del resultado anterior.

In [64]:
SQL_MAPA_06 = """
SELECT
    p.id,
    p.idcad,
    p.coop,
    p.capa_origen,
    ST_AsGeoJSON(ST_Transform(p.geom, 4326))::json AS geojson
FROM gis.mt_postes p
WHERE ST_Within(
    p.geom,
    ST_MakeEnvelope(720000, 7040000, 725000, 7045000, 32721)
);
"""

mostrar_mapa(ejecutar_geojson(SQL_MAPA_06), 'Consulta 6 — postes en ventana de trabajo')

# Bloque 3 — Resúmenes operativos

Este bloque resume longitudes y radios operativos. Una consulta es tabular; la otra vuelve al mapa.

## Consulta 7: Longitud total de cables por cooperativa

**Consulta espacial:** Medición de geometrías ST_Length  
**Función:** Calcular longitud de red.  
**Consigna:** Calcular la cantidad de tramos y la longitud total de cables por cooperativa.

**Idea clave:** Como geom está en SRID 32721, ST_Length devuelve la longitud en metros. La consulta resume la red por cooperativa.

In [65]:
SQL_CONSULTA_07 = """
SELECT
    c.coop,
    count(*) AS cantidad_tramos,
    round(sum(ST_Length(c.geom))::numeric, 2) AS longitud_total_m
FROM gis.mt_cables c
GROUP BY c.coop
ORDER BY longitud_total_m DESC;
"""

print(SQL_CONSULTA_07)


SELECT
    c.coop,
    count(*) AS cantidad_tramos,
    round(sum(ST_Length(c.geom))::numeric, 2) AS longitud_total_m
FROM gis.mt_cables c
GROUP BY c.coop
ORDER BY longitud_total_m DESC;



In [66]:
ejecutar_sql(SQL_CONSULTA_07)

,coop,cantidad_tramos,longitud_total_m
0,MCARL,7146,603468.70
1,NULL,11,0.00


## Consulta 8: Postes dentro del radio operativo de seccionadores

**Consulta espacial:** Proximidad entre capas ST_DWithin  
**Función:** Contar postes cercanos a seccionadores.  
**Consigna:** Para cada seccionador, contar cuántos postes hay dentro de un radio de 100 metros.

**Idea clave:** Relaciona seccionadores con postes cercanos. El LEFT JOIN permite conservar seccionadores aunque no tengan postes dentro del radio definido.

In [67]:
SQL_CONSULTA_08 = """
SELECT
    s.id,
    s.idcad,
    s.coop,
    count(p.id) AS postes_en_radio
FROM gis.mt_seccionadores s
LEFT JOIN gis.mt_postes p
    ON ST_DWithin(p.geom, s.geom, 100)
GROUP BY s.id, s.idcad, s.coop
ORDER BY postes_en_radio DESC;
"""

print(SQL_CONSULTA_08)


SELECT
    s.id,
    s.idcad,
    s.coop,
    count(p.id) AS postes_en_radio
FROM gis.mt_seccionadores s
LEFT JOIN gis.mt_postes p
    ON ST_DWithin(p.geom, s.geom, 100)
GROUP BY s.id, s.idcad, s.coop
ORDER BY postes_en_radio DESC;



In [68]:
ejecutar_sql(SQL_CONSULTA_08)

,id,idcad,coop,postes_en_radio
0,31,MTENS273021,MCARL,15
1,17,MTENS273281,MCARL,15
2,19,MTENS273277,MCARL,15
3,16,MTENS273282,MCARL,15
4,83,MTENS269360,MCARL,15
...,...,...,...,...
876,148,MTENS267203,MCARL,1
877,357,MTENS260444,MCARL,1
878,201,MTENS265755,MCARL,1
879,220,MTENS264868,MCARL,1


### Mapa consulta 8

Visualización espacial del resultado anterior.

In [69]:
SQL_MAPA_08 = """
SELECT
    s.id,
    s.idcad,
    s.coop,
    count(p.id) AS postes_en_radio,
    ST_AsGeoJSON(ST_Transform(ST_Buffer(s.geom, 100), 4326))::json AS geojson
FROM gis.mt_seccionadores s
LEFT JOIN gis.mt_postes p
    ON ST_DWithin(p.geom, s.geom, 100)
GROUP BY s.id, s.idcad, s.coop, s.geom
ORDER BY postes_en_radio DESC
LIMIT 25;
"""

mostrar_mapa(ejecutar_geojson(SQL_MAPA_08), 'Consulta 8 — radios operativos de seccionadores')

# Bloque 4 — Clustering y áreas de operación

Este bloque usa funciones más avanzadas: agrupamiento espacial y generación de áreas.

## Consulta 9: Agrupamiento espacial de postes

**Consulta espacial:** Clustering ST_ClusterDBSCAN  
**Función:** Detectar concentraciones de postes.  
**Consigna:** Agrupar postes que estén próximos entre sí, usando una distancia máxima de 80 metros y un mínimo de 5 postes por grupo.

**Idea clave:** ST_ClusterDBSCAN identifica grupos espaciales de postes próximos. eps := 80 define la distancia máxima entre vecinos y minpoints := 5 exige al menos cinco postes para formar grupo.

In [70]:
SQL_CONSULTA_09 = """
SELECT
    cid AS grupo,
    count(*) AS cantidad_postes
FROM (
    SELECT
        p.id,
        ST_ClusterDBSCAN(
            p.geom,
            eps := 80,
            minpoints := 5
        ) OVER () AS cid
    FROM gis.mt_postes p
) q
WHERE cid IS NOT NULL
GROUP BY cid
ORDER BY cantidad_postes DESC;
"""

print(SQL_CONSULTA_09)


SELECT
    cid AS grupo,
    count(*) AS cantidad_postes
FROM (
    SELECT
        p.id,
        ST_ClusterDBSCAN(
            p.geom,
            eps := 80,
            minpoints := 5
        ) OVER () AS cid
    FROM gis.mt_postes p
) q
WHERE cid IS NOT NULL
GROUP BY cid
ORDER BY cantidad_postes DESC;



In [71]:
ejecutar_sql(SQL_CONSULTA_09)

,grupo,cantidad_postes
0,79,118
1,39,115
2,41,80
3,70,26
4,36,24
...,...,...
90,63,4
91,45,4
92,59,3
93,89,3


### Mapa consulta 9

Visualización espacial del resultado anterior.

In [72]:
SQL_MAPA_09 = """
SELECT
    cid AS grupo,
    count(*) AS cantidad_postes,
    ST_AsGeoJSON(ST_Transform(ST_Collect(geom), 4326))::json AS geojson
FROM (
    SELECT
        p.id,
        p.geom,
        ST_ClusterDBSCAN(
            p.geom,
            eps := 80,
            minpoints := 5
        ) OVER () AS cid
    FROM gis.mt_postes p
) q
WHERE cid IS NOT NULL
GROUP BY cid
ORDER BY cantidad_postes DESC;
"""

mostrar_mapa(ejecutar_geojson(SQL_MAPA_09), 'Consulta 9 — grupos espaciales de postes')

## Consulta 10: Área de operación alrededor de seccionadores

**Consulta espacial:** Generación de áreas ST_Buffer  
**Función:** Crear zonas de influencia.  
**Consigna:** Generar un área de operación de 50 metros alrededor de cada seccionador.

**Idea clave:** ST_Buffer crea una geometría poligonal alrededor de cada seccionador. Como la capa está en 32721, el valor 50 representa 50 metros.

In [73]:
SQL_CONSULTA_10 = """
SELECT
    s.id,
    s.idcad,
    s.coop,
    round(ST_Area(ST_Buffer(s.geom, 50))::numeric, 2) AS area_m2,
    left(ST_AsText(ST_Buffer(s.geom, 50)), 160) || '...' AS area_operacion_wkt
FROM gis.mt_seccionadores s;
"""

print(SQL_CONSULTA_10)


SELECT
    s.id,
    s.idcad,
    s.coop,
    round(ST_Area(ST_Buffer(s.geom, 50))::numeric, 2) AS area_m2,
    left(ST_AsText(ST_Buffer(s.geom, 50)), 160) || '...' AS area_operacion_wkt
FROM gis.mt_seccionadores s;



In [74]:
ejecutar_sql(SQL_CONSULTA_10)

,id,idcad,coop,area_m2,area_operacion_wkt
0,1,MTENS459316,MCARL,7803.61,"POLYGON((726834.5548 7057570.136,726833.594064..."
1,2,MTENS241125,MCARL,7803.61,"POLYGON((726739.3877 7059036.2351,726738.42696..."
2,3,MTENS459288,MCARL,7803.61,"POLYGON((726417.0543 7059446.8145,726416.09356..."
3,4,MTENS459222,MCARL,7803.61,"POLYGON((728724.1472 7029621.1122,728723.18646..."
4,5,MTENS459220,MCARL,7803.61,"POLYGON((728534.05 7030727.9576,728533.0892640..."
...,...,...,...,...,...
876,877,MTENS211013,MCARL,7803.61,"POLYGON((724581.0218 7058398.8637,724580.06106..."
877,878,MTENS211012,MCARL,7803.61,"POLYGON((725309.5724 7058362.8698,725308.61166..."
878,879,MTENS211008,MCARL,7803.61,"POLYGON((725884.0979 7058474.8425,725883.13716..."
879,880,MTENS211006,MCARL,7803.61,"POLYGON((726235.3045 7058105.4697,726234.34376..."


### Mapa consulta 10

Visualización espacial del resultado anterior.

In [75]:
SQL_MAPA_10 = """
SELECT
    s.id,
    s.idcad,
    s.coop,
    round(ST_Area(ST_Buffer(s.geom, 50))::numeric, 2) AS area_m2,
    ST_AsGeoJSON(ST_Transform(ST_Buffer(s.geom, 50), 4326))::json AS geojson
FROM gis.mt_seccionadores s;
"""

mostrar_mapa(ejecutar_geojson(SQL_MAPA_10), 'Consulta 10 — áreas de operación')

# Cierre

La notebook reúne las 10 consultas espaciales MT. Cada consulta deja una salida tabular y, cuando corresponde, una visualización en mapa. Para trabajar en clase, conviene ejecutar bloque por bloque y revisar primero los resultados tabulares antes del mapa.